# Arms 5 & 6: heog_on and restEO Sensitivity Checks

**Objective.** Extract FAA (alpha power, F3/F4) from the two preprocessing variants not covered by `full_cohort_features.parquet` (restEC/heog_off only), then run both as FAA-alone comparisons - Arm 5 (heog_off vs. heog_on, Decision 1) and Arm 6 (restEC vs. restEO, Decision 4) - using the same harness as Arm 1. Combined into one notebook rather than split, given both are short, identical-in-structure reruns of Arm 1's exact procedure against different input data.

**Inputs.**
- Preprocessed epochs from `data/derivatives_heog_on/` (restEC condition) and `data/derivatives_heog_off/` (restEO condition)
- Age, responder status (`cohort_filtered_n163.xlsx`, unchanged from Arm 1)

**Feature construction.** Alpha power (8–13 Hz) at F3 and F4, via `compute_band_power` (`src/features.py`), same method as `full_cohort_features.parquet`'s original build. FAA = raw F4−F3, matching Arm 1 exactly, for both variants.

**Assumptions.**
- Subject usability checked explicitly per variant/condition against the QC log before extraction, rather than assumed to match Arm 1's 160 - confirmed identical (160 "ok" subjects, 3 errors, for every combination).
- Both arms reuse `AgeDeconfounder`, `run_nested_cv`, and the permutation-test pattern from `src/modelling.py` unchanged - same four classifiers, same class-balanced weighting, same `N_OUTER_SPLITS`/`N_INNER_SPLITS`/`RANDOM_STATE` as Arm 1, so results are directly comparable to Arm 1's numbers.
- These are sensitivity checks on Arm 1's already-finalized result, not new hypotheses - the question each answers is "does this preprocessing choice change the FAA finding," not "is there a new effect here."

In [14]:
# Imports
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing import find_repo_root
from src.features import load_subject_epochs, get_subject_qc, compute_band_power

project_root = find_repo_root()
data_dir = project_root / "data"

# QC log already covers all conditions/variants that were preprocessed —
# confirm what's actually available before assuming it matches Arm 1's 160
qc_log = pd.read_csv(data_dir / "batch_results_log_full_cohort.csv")
print(qc_log[['condition', 'heog_variant', 'status']].value_counts())

condition  heog_variant  status
restEC     heog_off      ok        160
restEO     heog_off      ok        160
restEC     heog_on       ok        160
restEO     heog_on       ok        160
restEC     heog_off      error       3
restEO     heog_off      error       3
restEC     heog_on       error       3
restEO     heog_on       error       3
Name: count, dtype: int64


In [6]:
# Restrict to subjects with status == 'ok' for each specific combination
arm5_subjects = qc_log[(qc_log['condition'] == 'restEC') &
                       (qc_log['heog_variant'] == 'heog_on') &
                       (qc_log['status'] == 'ok')]['subject_id'].tolist()
arm6_subjects = qc_log[(qc_log['condition'] == 'restEO') &
                       (qc_log['heog_variant'] == 'heog_off') &
                       (qc_log['status'] == 'ok')]['subject_id'].tolist()

print(len(arm5_subjects), len(arm6_subjects))  # expect 160, 160

160 160


In [7]:
# Extraction: alpha power at F3, F4, for both new condition/variant combinations
ALPHA_BAND = {'alpha': (8, 13)}  # Decision 2

def extract_alpha_power(condition, variant, subject_ids, data_dir):
    """Extract F3/F4 alpha power for one condition/variant combination."""
    rows = []
    for subject_id in subject_ids:
        epochs = load_subject_epochs(subject_id, condition, variant, data_dir)
        band_power = compute_band_power(epochs, ALPHA_BAND)
        rows.append({
            'subject_id': subject_id,
            'F3_alpha_power': band_power['F3_alpha_power'],
            'F4_alpha_power': band_power['F4_alpha_power'],
        })
    return pd.DataFrame(rows)

arm5_alpha = extract_alpha_power('restEC', 'heog_on', arm5_subjects, data_dir)
arm6_alpha = extract_alpha_power('restEO', 'heog_off', arm6_subjects, data_dir)

print(arm5_alpha.shape, arm6_alpha.shape)  # expect (160, 3) each
arm5_alpha.head()

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_on/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_on/sub-88049537/sub-88049537_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_on/sub-88049857/sub-88049857_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensatio

,subject_id,F3_alpha_power,F4_alpha_power
0,sub-87999321,122.985929,120.844889
1,sub-88049537,36.712893,33.292931
2,sub-88049857,212.159853,203.117107
3,sub-88049905,144.841124,142.835074
4,sub-88050713,319.781932,327.519484


In [8]:
# Build FAA and merge with age/Responder, for both arms
cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")

def build_model_df(alpha_df, cohort_df):
    """FAA + age + Responder, from a raw alpha-power extraction."""
    model_df = pd.DataFrame({
        'subject_id': alpha_df['subject_id'],
        'FAA': alpha_df['F4_alpha_power'] - alpha_df['F3_alpha_power'],
    })
    model_df = model_df.merge(
        cohort_df[['TDBRAIN_ID', 'age', 'Responder']],
        left_on='subject_id', right_on='TDBRAIN_ID', how='left'
    ).drop(columns='TDBRAIN_ID')
    return model_df

arm5_model_df = build_model_df(arm5_alpha, cohort_df)
arm6_model_df = build_model_df(arm6_alpha, cohort_df)

for name, df in [('Arm 5 (heog_on)', arm5_model_df), ('Arm 6 (restEO)', arm6_model_df)]:
    print(name)
    print(len(df))                                          # must stay 160
    print(df[['FAA', 'age', 'Responder']].isna().sum())      # must all be 0
    print(df['Responder'].value_counts())                    # expect ~93/67
    print()

Arm 5 (heog_on)
160
FAA          0
age          0
Responder    0
dtype: int64
Responder
1    93
0    67
Name: count, dtype: int64

Arm 6 (restEO)
160
FAA          0
age          0
Responder    0
dtype: int64
Responder
1    93
0    67
Name: count, dtype: int64



In [9]:
# Save for the Arm 5 / Arm 6 modelling 
features_dir = data_dir / "features"
arm5_model_df.to_parquet(features_dir / "arm5_heog_on_faa.parquet")
arm6_model_df.to_parquet(features_dir / "arm6_restEO_faa.parquet")

print("saved:", (features_dir / "arm5_heog_on_faa.parquet").exists(),
                (features_dir / "arm6_restEO_faa.parquet").exists())

saved: True True


In [10]:
## Arm 5: heog_off vs. heog_on (FAA alone)

from src.modelling import AgeDeconfounder, run_nested_cv, paired_comparison_test
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

N_OUTER_SPLITS = 5
N_INNER_SPLITS = 3
RANDOM_STATE = 42

X_arm5 = arm5_model_df[['FAA']].values
y_arm5 = arm5_model_df['Responder'].values
age_arm5 = arm5_model_df['age'].values

classifier_specs_balanced = {
    'LDA (Ledoit-Wolf, balanced priors)': (
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto', priors=[0.5, 0.5]),
        None
    ),
    'Logistic (unregularized, balanced)': (
        LogisticRegression(C=np.inf, max_iter=1000, class_weight='balanced'),
        None
    ),
    'Logistic (elastic-net, balanced)': (
        LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
    ),
    'Logistic (L2 / "Bayesian" MAP, balanced)': (
        LogisticRegression(l1_ratio=0, max_iter=1000, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
}

arm5_results_df = run_nested_cv(X_arm5, y_arm5, age_arm5, classifier_specs_balanced,
                                 N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)

print(arm5_results_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
arm5_results_df

                                          balanced_accuracy  accuracy  \
classifier                                                              
LDA (Ledoit-Wolf, balanced priors)                 0.576730   0.59375   
Logistic (L2 / "Bayesian" MAP, balanced)           0.584423   0.60000   
Logistic (elastic-net, balanced)                   0.584423   0.60000   
Logistic (unregularized, balanced)                 0.584423   0.60000   

                                               auc  sensitivity  specificity  \
classifier                                                                     
LDA (Ledoit-Wolf, balanced priors)        0.618807     0.675439     0.478022   
Logistic (L2 / "Bayesian" MAP, balanced)  0.618807     0.675439     0.493407   
Logistic (elastic-net, balanced)          0.618807     0.675439     0.493407   
Logistic (unregularized, balanced)        0.618807     0.675439     0.493407   

                                               ppv  
classifier                 

,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv
0,"LDA (Ledoit-Wolf, balanced priors)",0,0.585020,0.59375,0.595142,0.631579,0.538462,0.666667
1,"Logistic (unregularized, balanced)",0,0.585020,0.59375,0.595142,0.631579,0.538462,0.666667
2,"Logistic (elastic-net, balanced)",0,0.585020,0.59375,0.595142,0.631579,0.538462,0.666667
3,"Logistic (L2 / ""Bayesian"" MAP, balanced)",0,0.585020,0.59375,0.595142,0.631579,0.538462,0.666667
4,"LDA (Ledoit-Wolf, balanced priors)",1,0.637652,0.65625,0.611336,0.736842,0.538462,0.700000
5,"Logistic (unregularized, balanced)",1,0.637652,0.65625,0.611336,0.736842,0.538462,0.700000
6,"Logistic (elastic-net, balanced)",1,0.637652,0.65625,0.611336,0.736842,0.538462,0.700000
7,"Logistic (L2 / ""Bayesian"" MAP, balanced)",1,0.637652,0.65625,0.611336,0.736842,0.538462,0.700000
8,"LDA (Ledoit-Wolf, balanced priors)",2,0.613360,0.65625,0.732794,0.842105,0.384615,0.666667
9,"Logistic (unregularized, balanced)",2,0.651822,0.68750,0.732794,0.842105,0.461538,0.695652


In [15]:
# Diagnostic: confirm the elastic-net/L2/unregularized tie is real convergence
fold_idx_to_check = 0
outer_cv_check = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = list(outer_cv_check.split(X_arm5, y_arm5))[fold_idx_to_check]

X_train, X_test = X_arm5[train_idx], X_arm5[test_idx]
y_train, y_test = y_arm5[train_idx], y_arm5[test_idx]
age_train, age_test = age_arm5[train_idx], age_arm5[test_idx]

deconf = AgeDeconfounder()
deconf.fit(X_train, age_train)
X_train_clean = deconf.transform(X_train, age_train)
X_test_clean = deconf.transform(X_test, age_test)

scaler = StandardScaler()
X_train_clean = scaler.fit_transform(X_train_clean)
X_test_clean = scaler.transform(X_test_clean)

for clf_name, (estimator, param_grid) in classifier_specs_balanced.items():
    if param_grid is not None:
        inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(estimator, param_grid, cv=inner_cv, scoring='balanced_accuracy')
        search.fit(X_train_clean, y_train)
        fitted_model = search.best_estimator_
        print(clf_name, "best params:", search.best_params_)
    else:
        fitted_model = estimator.fit(X_train_clean, y_train)

    proba = fitted_model.predict_proba(X_test_clean)[:, 1]
    print(clf_name)
    print("  probabilities:", np.round(proba, 3))
    print()

LDA (Ledoit-Wolf, balanced priors)
  probabilities: [0.471 0.379 0.405 0.509 0.533 0.497 0.36  0.528 0.578 0.568 0.405 0.321
 0.376 0.502 0.36  0.533 0.517 0.403 0.534 0.588 0.557 0.579 0.403 0.562
 0.483 0.51  0.525 0.527 0.495 0.443 0.532 0.523]

Logistic (unregularized, balanced)
  probabilities: [0.47  0.372 0.399 0.51  0.535 0.497 0.352 0.53  0.583 0.573 0.398 0.311
 0.368 0.503 0.351 0.535 0.518 0.396 0.537 0.595 0.561 0.585 0.397 0.567
 0.482 0.511 0.527 0.529 0.495 0.439 0.534 0.525]

Logistic (elastic-net, balanced) best params: {'C': 0.1, 'l1_ratio': 0.1}
Logistic (elastic-net, balanced)
  probabilities: [0.48  0.413 0.432 0.506 0.523 0.498 0.399 0.52  0.556 0.549 0.432 0.37
 0.41  0.502 0.399 0.524 0.512 0.43  0.524 0.563 0.541 0.557 0.43  0.544
 0.488 0.507 0.518 0.519 0.497 0.459 0.523 0.516]

Logistic (L2 / "Bayesian" MAP, balanced) best params: {'C': 0.01}
Logistic (L2 / "Bayesian" MAP, balanced)
  probabilities: [0.493 0.471 0.477 0.502 0.508 0.499 0.467 0.506 0.518 0.5

In [16]:
# Permutation test: Arm 5 (heog_on)
N_PERMUTATIONS = 1000
rng = np.random.RandomState(RANDOM_STATE)

observed_arm5 = arm5_results_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

print(f"N_PERMUTATIONS = {N_PERMUTATIONS}")

null_scores_arm5 = {name: [] for name in classifier_specs_balanced}

for i in range(N_PERMUTATIONS):
    y_shuffled = rng.permutation(y_arm5)
    perm_df = run_nested_cv(X_arm5, y_shuffled, age_arm5, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_result = perm_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()
    for name, score in perm_result.items():
        null_scores_arm5[name].append(score)

print(f"{'classifier':<45} {'observed':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_balanced:
    null_arr = np.array(null_scores_arm5[name])
    p_value = (np.sum(null_arr >= observed_arm5[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed_arm5[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

N_PERMUTATIONS = 1000
classifier                                      observed  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.5767     0.5000     0.0370
Logistic (unregularized, balanced)                0.5844     0.4999     0.0250
Logistic (elastic-net, balanced)                  0.5844     0.4999     0.0190
Logistic (L2 / "Bayesian" MAP, balanced)          0.5844     0.4998     0.0260


## Arm 5 (heog_on) - Findings

Arm 1 used `heog_off` (no HEOG correction) as the primary preprocessing choice. Arm 5 re-tests the same FAA feature and pipeline under `heog_on` (HEOG correction applied), as a sensitivity check on that preprocessing decision.

Permutation test (N=1000) against chance-level balanced accuracy, same nested-CV configuration as the observed run (5 outer / 3 inner folds, random_state=42):

| Classifier | Observed BA | Null mean | p-value |
|---|---|---|---|
| LDA (Ledoit-Wolf, balanced priors) | 0.577 | 0.500 | 0.037 |
| Logistic (unregularized, balanced) | 0.584 | 0.500 | 0.025 |
| Logistic (elastic-net, balanced) | 0.584 | 0.500 | 0.019 |
| Logistic (L2, balanced) | 0.584 | 0.500 | 0.026 |

All four classifiers exceed chance at α=0.05. Null means cluster tightly around 0.50 across all classifiers, consistent with a correctly calibrated permutation procedure.

The near-identical observed scores among unregularized/elastic-net/L2 logistic variants mirror the pattern already diagnosed in Arm 1 (genuine convergence to similar solutions under this feature set, not a bug). Re-verified directly here (diagnostic cell above): raw predicted probabilities for one outer fold differ meaningfully between classifiers despite the tied balanced accuracy, confirming genuine convergence rather than a shared-grid-corner artifact.

Arm 5's results are similar in magnitude and significance to Arm 1's (elastic-net 0.587, p = 0.016; other three at the 0.05 boundary), indicating the FAA-responder association is not an artifact of the HEOG-correction choice - it holds under both variants.

**Residual assumptions**

No direct statistical comparison to Arm 1 has been run; magnitudes are similar but "replicates" or "improves on" claims about Arm 1 require `paired_comparison_test`, not this result.

The probability-convergence check above was run on one outer fold only, not all five; treated as sufficient given the clear numerical evidence, but not exhaustively verified.

This check addresses the HEOG-correction decision only. Arm 1's open seed-sensitivity question (results checked only at `random_state=42`, not tested across seeds) remains unresolved and applies equally here.

In [18]:
## Arm 6: restEC vs. restEO (FAA alone)

X_arm6 = arm6_model_df[['FAA']].values
y_arm6 = arm6_model_df['Responder'].values
age_arm6 = arm6_model_df['age'].values

# classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE
# already defined above (from Arm 5) - reused as-is, per the plan

arm6_results_df = run_nested_cv(X_arm6, y_arm6, age_arm6, classifier_specs_balanced,
                                 N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
print(arm6_results_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
arm6_results_df

                                          balanced_accuracy  accuracy  \
classifier                                                              
LDA (Ledoit-Wolf, balanced priors)                 0.608007    0.6125   
Logistic (L2 / "Bayesian" MAP, balanced)           0.608007    0.6125   
Logistic (elastic-net, balanced)                   0.608007    0.6125   
Logistic (unregularized, balanced)                 0.608007    0.6125   

                                               auc  sensitivity  specificity  \
classifier                                                                     
LDA (Ledoit-Wolf, balanced priors)        0.621772     0.646784     0.569231   
Logistic (L2 / "Bayesian" MAP, balanced)  0.621772     0.646784     0.569231   
Logistic (elastic-net, balanced)          0.621772     0.646784     0.569231   
Logistic (unregularized, balanced)        0.621772     0.646784     0.569231   

                                               ppv  
classifier                 

,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv
0,"LDA (Ledoit-Wolf, balanced priors)",0,0.625506,0.65625,0.740891,0.789474,0.461538,0.681818
1,"Logistic (unregularized, balanced)",0,0.625506,0.65625,0.740891,0.789474,0.461538,0.681818
2,"Logistic (elastic-net, balanced)",0,0.625506,0.65625,0.740891,0.789474,0.461538,0.681818
3,"Logistic (L2 / ""Bayesian"" MAP, balanced)",0,0.625506,0.65625,0.740891,0.789474,0.461538,0.681818
4,"LDA (Ledoit-Wolf, balanced priors)",1,0.532389,0.53125,0.502024,0.526316,0.538462,0.625000
5,"Logistic (unregularized, balanced)",1,0.532389,0.53125,0.502024,0.526316,0.538462,0.625000
6,"Logistic (elastic-net, balanced)",1,0.532389,0.53125,0.502024,0.526316,0.538462,0.625000
7,"Logistic (L2 / ""Bayesian"" MAP, balanced)",1,0.532389,0.53125,0.502024,0.526316,0.538462,0.625000
8,"LDA (Ledoit-Wolf, balanced priors)",2,0.659919,0.62500,0.643725,0.473684,0.846154,0.818182
9,"Logistic (unregularized, balanced)",2,0.659919,0.62500,0.643725,0.473684,0.846154,0.818182


### Permutation test: Arm 6 (restEO)

Same procedure as Arm 5: shuffle `y_arm6` 1000 times, rerunning the full nested-CV pipeline (age deconfounding, scaling, inner-loop tuning, outer scoring) on each shuffle to build a null distribution of balanced accuracy per classifier. `age_arm6` is not shuffled - only the label-feature relationship is being broken. Same nested-CV configuration as the Arm 6 observed run (5 outer / 3 inner folds, `random_state=42`), so the null is comparable to `observed_arm6`.

This tests Arm 6 (restEO) against chance, not against Arm 1 or Arm 5 — no cross-arm comparison is performed here.

In [19]:
# Permutation test: Arm 6 (restEO)
N_PERMUTATIONS = 1000
rng = np.random.RandomState(RANDOM_STATE)

observed_arm6 = arm6_results_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

print(f"N_PERMUTATIONS = {N_PERMUTATIONS}")

null_scores_arm6 = {name: [] for name in classifier_specs_balanced}

for i in range(N_PERMUTATIONS):
    y_shuffled = rng.permutation(y_arm6)
    perm_df = run_nested_cv(X_arm6, y_shuffled, age_arm6, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_result = perm_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()
    for name, score in perm_result.items():
        null_scores_arm6[name].append(score)

print(f"{'classifier':<45} {'observed':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_balanced:
    null_arr = np.array(null_scores_arm6[name])
    p_value = (np.sum(null_arr >= observed_arm6[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed_arm6[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

N_PERMUTATIONS = 1000
classifier                                      observed  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.6080     0.5016     0.0070
Logistic (unregularized, balanced)                0.6080     0.5015     0.0070
Logistic (elastic-net, balanced)                  0.6080     0.5012     0.0050
Logistic (L2 / "Bayesian" MAP, balanced)          0.6080     0.5016     0.0070


In [20]:
# Sanity check: are LDA and the three logistic variants producing genuinely
# different probabilities, or is the 0.6080 tie across all four suspicious?
# Same diagnostic as Arm 1's probability-inspection cell, applied to Arm 6.

outer_cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_cv.split(X_arm6, y_arm6))

X_train, X_test = X_arm6[train_idx], X_arm6[test_idx]
y_train, y_test = y_arm6[train_idx], y_arm6[test_idx]
age_train, age_test = age_arm6[train_idx], age_arm6[test_idx]

deconf = AgeDeconfounder()
deconf.fit(X_train, age_train)
X_train_clean = deconf.transform(X_train, age_train)
X_test_clean = deconf.transform(X_test, age_test)

for clf_name, (estimator, param_grid) in classifier_specs_balanced.items():
    if param_grid is not None:
        inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(estimator, param_grid, cv=inner_cv, scoring='balanced_accuracy')
        search.fit(X_train_clean, y_train)
        fitted_model = search.best_estimator_
        print(clf_name, "best params:", search.best_params_)
    else:
        fitted_model = estimator.fit(X_train_clean, y_train)

    proba = fitted_model.predict_proba(X_test_clean)[:, 1]
    preds = fitted_model.predict(X_test_clean)
    print(clf_name)
    print("  probabilities:", np.round(proba, 3))
    print("  predictions:  ", preds)
    print()

print("y_test:              ", y_test)

LDA (Ledoit-Wolf, balanced priors)
  probabilities: [0.525 0.523 0.405 0.503 0.528 0.503 0.497 0.52  0.536 0.501 0.491 0.448
 0.428 0.48  0.493 0.522 0.506 0.527 0.512 0.608 0.511 0.592 0.462 0.494
 0.567 0.53  0.547 0.505 0.51  0.36  0.51  0.535]
  predictions:   [1 1 0 1 1 1 0 1 1 1 0 0 0 0 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1]

Logistic (unregularized, balanced)
  probabilities: [0.526 0.524 0.402 0.503 0.529 0.504 0.497 0.521 0.537 0.502 0.491 0.446
 0.425 0.479 0.493 0.523 0.506 0.528 0.512 0.612 0.512 0.596 0.46  0.494
 0.57  0.531 0.549 0.506 0.51  0.355 0.51  0.537]
  predictions:   [1 1 0 1 1 1 0 1 1 1 0 0 0 0 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1]

Logistic (elastic-net, balanced) best params: {'C': 1, 'l1_ratio': 0.1}
Logistic (elastic-net, balanced)
  probabilities: [0.526 0.524 0.402 0.503 0.529 0.504 0.497 0.521 0.537 0.502 0.491 0.446
 0.425 0.479 0.493 0.523 0.507 0.528 0.512 0.612 0.512 0.596 0.461 0.494
 0.57  0.531 0.549 0.506 0.51  0.355 0.51  0.537]
  predictions:   [

## Arm 6 (restEO) - Findings

Arm 1 used `restEC` (eyes-closed) as the primary resting condition. Arm 6 re-tests the same FAA feature and pipeline under `restEO` (eyes-open), holding HEOG correction fixed at `heog_off` (matching Arm 1) - isolating the resting-condition choice as the single sensitivity axis, separate from Arm 5's HEOG-correction axis.

Permutation test (N=1000) against chance-level balanced accuracy, same nested-CV configuration as the observed run (5 outer / 3 inner folds, random_state=42):

| Classifier | Observed BA | Null mean | p-value |
|---|---|---|---|
| LDA (Ledoit-Wolf, balanced priors) | 0.608 | 0.502 | 0.007 |
| Logistic (unregularized, balanced) | 0.608 | 0.502 | 0.007 |
| Logistic (elastic-net, balanced) | 0.608 | 0.501 | 0.005 |
| Logistic (L2, balanced) | 0.608 | 0.502 | 0.007 |

All four classifiers exceed chance at α=0.05, with stronger p-values than either Arm 1 or Arm 5. Null means cluster tightly around 0.50, consistent with a correctly calibrated permutation procedure.

All four classifiers - including LDA, not just the three logistic variants as in Arm 1/Arm 5 - converge to identical balanced accuracy (0.6080) across every outer fold. Checked directly against raw predicted probabilities (not just predictions) for one fold: probabilities differ meaningfully between classifiers (e.g. 0.405/0.402/0.402/0.408 for one subject), and elastic-net selected weak regularization (`C=1, l1_ratio=0.1`) rather than L2's near-maximal shrinkage (`C=0.01`) - so this is not the same "shared grid corner" mechanism diagnosed in Arm 1. The more likely explanation is that with a single predictor (FAA), any reasonable classifier draws a similar one-dimensional decision boundary regardless of algorithm or regularization strength - the balanced-accuracy analogue of the identical-AUC effect already noted in Arm 1's summary. Confirmed genuine convergence, not a bug, though only directly checked on one fold.

Arm 6's balanced accuracy (0.608) and p-values (0.005-0.007) are numerically stronger than both Arm 1 (0.571-0.587, p = 0.016-0.052) and Arm 5 (0.577-0.584, p = 0.019-0.037).

**Residual assumptions**

No direct statistical comparison to Arm 1 or Arm 5 has been run; "stronger" is a descriptive observation about these specific numbers, not a tested claim - a formal comparison would require `paired_comparison_test`.

The probability-convergence check was run on one outer fold only, not all five; treated as sufficient given the clear numerical evidence (distinct probability values, differing hyperparameters, identical classifications), but not exhaustively verified across folds.

This check addresses the resting-condition (restEC vs. restEO) decision only. Arm 1's open seed-sensitivity question (results checked only at `random_state=42`) remains unresolved and applies equally here.